In [ ]:
import cv2
import numpy as np

import mediapipe as mp
from mediapipe.tasks.python.vision import (
    drawing_utils, drawing_styles,
    HandLandmarker, HandLandmarksConnections, HandLandmarkerOptions,
)

In [2]:
model_path = '../models/hand_landmarker.task'
hand_opt = HandLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(model_asset_path=model_path),
    num_hands=2,
    min_hand_detection_confidence=0.6,
    min_hand_presence_confidence=0.6,
    min_tracking_confidence=0.6
)
hand_detector = HandLandmarker.create_from_options(hand_opt)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1786550712.152630   43504 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1786550712.165306   43503 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [3]:
def draw_landmarks(frame, hand_landmarks):
    drawing_utils.draw_landmarks(
        frame,
        hand_landmarks,
        HandLandmarksConnections.HAND_CONNECTIONS,
        drawing_utils.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
        drawing_utils.DrawingSpec(color=(255, 0, 0), thickness=2)
    )

In [4]:
def distance(a, b):
    return ((a.x - b.x)**2 + (a.y - b.y)**2) ** 0.5

In [5]:
image = mp.Image.create_from_file('../models/woman_hands.jpg')
result = hand_detector.detect(image)

W0000 00:00:1786550712.217127   43504 landmark_projection_calculator.cc:81] Using NORM_RECT without IMAGE_DIMENSIONS is only supported for the square ROI. Provide IMAGE_DIMENSIONS or use PROJECTION_MATRIX.


In [6]:
for hand_landmarks in result.hand_landmarks:
    thumb_tip = hand_landmarks[4]
    index_tip = hand_landmarks[8]

    dst = distance(thumb_tip, index_tip)
    print(dst)

0.09314065518399156
0.05942703722545447


In [7]:
def landmark_to_pixel(landmark, frame_shape):
    h, w = frame_shape[:2]

    return (
        int(landmark.x * w),
        int(landmark.y * h)
    )

def draw_area(results, frame_shape):
    if len(results.hand_landmarks) < 2:
        return

    left_hand = None
    right_hand = None

    for landmarks, handedness in zip(
        results.hand_landmarks, results.handedness
    ):
        hand_name = handedness[0].category_name

        if hand_name == "Left":
            left_hand = landmarks
        elif hand_name == "Right":
            right_hand = landmarks

    if left_hand is None or right_hand is None:
        return

    left_thumb = landmark_to_pixel(left_hand[4], frame_shape)
    left_index = landmark_to_pixel(left_hand[8], frame_shape)
    right_thumb = landmark_to_pixel(right_hand[4], frame_shape)
    right_index = landmark_to_pixel(right_hand[8], frame_shape)

    points = np.array([
        left_thumb, left_index, right_index, right_thumb
    ], dtype=np.int32)
    return points

In [8]:
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("No camera detected")

while True:
    ok, frame = cap.read()
    if not ok:
        break

    frame = cv2.flip(frame, 1)
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(mp.ImageFormat.SRGB, rgb)
    results = hand_detector.detect(mp_image)

    for hand_landmarks in results.hand_landmarks:
        draw_landmarks(frame, hand_landmarks)

        thumb_tip = hand_landmarks[4]
        index_tip = hand_landmarks[8]

        dst = distance(thumb_tip, index_tip)
        if dst < 0.05:
            cv2.putText(frame, "pinch", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

    points = draw_area(results, frame.shape)
    if points is not None:
        cv2.polylines(
            frame,
            [points],
            isClosed=True,
            color=(0, 255, 0),
            thickness=2
        )

    cv2.imshow("", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

QFontDatabase: Cannot find font directory /home/duke/miniconda3/envs/filter/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/duke/miniconda3/envs/filter/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/duke/miniconda3/envs/filter/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/duke/miniconda3/envs/filter/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find f